In [5]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import glob
from collections import defaultdict
import numpy as np
from matplotlib.patches import Rectangle
import sys
from pathlib import Path
if str(Path().resolve().parent) not in sys.path:
    sys.path.insert(0, str(Path().resolve().parent))
import json


# ИЗМЕНИТЬ ДЛЯ ИСПОЛЬЗОВАНИЯ
BASE_DIR = Path.cwd().parent
DATASET_PATH = BASE_DIR  / "dataset"
TRAIN_PATH = DATASET_PATH / "train"
TRAIN_PATH.exists()

True

In [6]:
# ИЗМЕНИТЬ ДЛЯ ИСПОЛЬЗОВАНИЯ
PROJECT_PATH = Path.cwd().parent
GENERAL_PATH = PROJECT_PATH / "coco_yolo"
TEST_PATH =    PROJECT_PATH / "dataset/test"
WEIGHTS_PATH = PROJECT_PATH / "model/runs/weights"

In [7]:
files = os.listdir(TRAIN_PATH)
is_json = lambda x: x.endswith("json")
is_img = lambda x: x.endswith("jpg")
coco_json = next(filter(is_json, files))

with open(f"{TRAIN_PATH}/{coco_json}") as coco_file:
    coco_json = json.load(coco_file)
    coco_annotations = coco_json.get("annotations")

categories = coco_json['categories']
category_map = {cat['id']: cat['name'] for cat in categories}
category_counts = {cat['id']: 0 for cat in categories}
for ann in coco_json['annotations']:
    category_counts[ann['category_id']] += 1

class_distribution = pd.DataFrame([
    {'category_id': cat_id, 'category_name': category_map[cat_id], 'count': count} 
    for cat_id, count in category_counts.items()
])

# Sort by count
class_distribution = class_distribution.sort_values('count', ascending=False)
class_distribution

,category_id,category_name,count
4,4,player,12261
5,5,referee,1321
1,1,ball,739
3,3,goalkeeper,281
2,2,coach,202
0,0,football-objects,0


In [8]:
[{'id': 0, 'name': 'football-objects', 'supercategory': 'none'},
 {'id': 1, 'name': 'ball', 'supercategory': 'football-objects'},
 {'id': 2, 'name': 'coach', 'supercategory': 'football-objects'},
 {'id': 3, 'name': 'goalkeeper', 'supercategory': 'football-objects'},
 {'id': 4, 'name': 'player', 'supercategory': 'football-objects'},
 {'id': 5, 'name': 'referee', 'supercategory': 'football-objects'}]

labels = {1: 'ball', 2: 'coach', 3: 'goalkeeper', 4: 'player', 5: 'referee'}

def process_labels_folder(labels_dir):
    list_w, list_h = [], []
    class_total_area = defaultdict(float)
    class_total_w = defaultdict(float)
    class_total_h = defaultdict(float)
    class_count = defaultdict(int)

    # Поиск всех .txt файлов в папке labels
    label_files = glob.glob(os.path.join(labels_dir, "*.txt"))

    for file_path in label_files:
        with open(file_path, 'r') as f:
            lines = f.readlines()
        for line in lines:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) < 5:
                continue

            try:
                cls = int(parts[0])
                w = float(parts[3])
                h = float(parts[4])
                area = w * h

                class_total_area[cls] += area
                class_total_w[cls] += w
                class_total_h[cls] += h
                list_w.append(w)
                list_h.append(h)
                class_count[cls] += 1

            except (ValueError, IndexError):
                print(f"Warning: некорректная строка в {file_path}: {line}")
                continue

    print("Статистика ширины и высота боксов по классам:")
    for cls in sorted(class_total_area.keys()):
        avg_area = class_total_area[cls] / class_count[cls]
        avg_w = class_total_w[cls] / class_count[cls]
        avg_h = class_total_h[cls] / class_count[cls]
        print(f"Класс {labels[cls]:<11} avg_w = {avg_w:.4f}, avg_h = {avg_h:.4f}, (всего боксов: {class_count[cls]})")

    np_list_w = np.array(list_w)
    np_list_h = np.array(list_h)

    print("\nОбщая статистика ширины и высота боксов:")
    print(f"min_w = {np_list_w.min()}, min_h = {np_list_h.min()}")
    print(f"max_w = {np_list_w.max()}, max_h = {np_list_h.max()}")
    print(f"quantile_75 = {np.quantile(np_list_h, 0.75):.4f}")
    print(f"quantile_85 = {np.quantile(np_list_h, 0.85):.4f}")
    print(f"quantile_95 = {np.quantile(np_list_h, 0.95):.4f}")
    print(f"quantile_99 = {np.quantile(np_list_h, 0.99):.4f}")

labels_dir = GENERAL_PATH / "labels/train"
if not os.path.isdir(labels_dir):
    print(f"Ошибка: папка '{labels_dir}' не найдена.")
    exit(1)
process_labels_folder(labels_dir)

Статистика ширины и высота боксов по классам:
Класс ball        avg_w = 0.0086, avg_h = 0.0156, (всего боксов: 591)
Класс coach       avg_w = 0.0245, avg_h = 0.0834, (всего боксов: 164)
Класс goalkeeper  avg_w = 0.0236, avg_h = 0.0785, (всего боксов: 219)
Класс player      avg_w = 0.0231, avg_h = 0.0850, (всего боксов: 9901)
Класс referee     avg_w = 0.0218, avg_h = 0.0766, (всего боксов: 1088)

Общая статистика ширины и высота боксов:
min_w = 0.00038, min_h = 0.000556
max_w = 0.087818, max_h = 0.351852
quantile_75 = 0.0955
quantile_85 = 0.1033
quantile_95 = 0.1188
quantile_99 = 0.1415
